# Preliminary evaluations of math removal and embedding models
This file tests the quality of embeddings across a few different models and styles of math removal. The models tested will be
* BM25 (baseline, not technically a model)
* SPECTER
* all-MiniLM-L12-v2
We will test against 3 types of math removal: the total math removal where all inline math is replaced with "math expression", the coarse removal where certain handwritten rules are applied and some math is replaced with its english meaning, and all other math is replaced with "math expression", and the fine removal where most math is compactified into one long word, for example "x + y > 10" becomes "x_plus_y_gt_10". 

In [1]:
from pathlib import Path
import sys

# annoying boilerplate that adds ../paper-clustering/ to the $PYTHONPATH variable 
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(sys.path)

from tqdm import tqdm
from supabase import create_client, Client
import pandas as pd
from pprint import pp

['/home/alfred/git/python/paper-clustering', '/home/alfred/miniconda3/envs/paper-clustering/lib/python311.zip', '/home/alfred/miniconda3/envs/paper-clustering/lib/python3.11', '/home/alfred/miniconda3/envs/paper-clustering/lib/python3.11/lib-dynload', '', '/home/alfred/miniconda3/envs/paper-clustering/lib/python3.11/site-packages']


## Collecting the data
In order to make sure that the result of the evaluation is meaningful, we want a dataset with a lot of hard examples, and that has a relatively wide range of resarch areas. In particular, we want to include pairs of papers with:
* Same topic, same technique
* Same topic, different technique
* Different topic, same technique
* Survey articles vs research articles
* Math heavy notations, light on math notation
* Same topic, different notations
And we also want a fairly wide range of topics so that we test the ability to find connections between topics in different areas, not just differentiate between papers in the same area.

In [2]:
CATEGORIES = ["cs.DS", "cs.IT", "cs.CC", "math.CO"]
DAYS_BACK = 10 * 365
from src.arxiv_fetch import fetch_arxiv_data
from src.logger import IngestionLogger
import logging
logging.getLogger().setLevel(logging.INFO)
logger = IngestionLogger("../failure.jsonl", "../skipped_papers.txt")

papers = []
ALL_KEYWORDS = []

In [ ]:
def collect_locality_papers(papers, ALL_KEYWORDS):
    # Collect some examples on locality in coding theory
    KEYWORDS = [
        "locally decodable",
        "locally correctable",
        "locally testable",
        "private information retrieval",
        "smooth code",
        "locally recoverable",
    ]
    ALL_KEYWORDS += KEYWORDS
    locality_papers = fetch_arxiv_data(CATEGORIES, 150, logger, DAYS_BACK, KEYWORDS)
    papers += locality_papers

In [4]:
def collect_coding_papers(papers, ALL_KEYWORDS):
    # Collect some examples on general coding theory
    KEYWORDS = [
        "erasure coding",
        "list decoding",
        "list recovery",
        "random code",
        "matching vector"
    ]
    ALL_KEYWORDS += KEYWORDS
    coding_papers = fetch_arxiv_data(CATEGORIES, 75, logger, DAYS_BACK, KEYWORDS)
    papers += coding_papers

In [5]:
def collect_itcr_papers(papers, ALL_KEYWORDS):   
     # Collect some examples in information theoretic crypto
    KEYWORDS = [
        "private information retrieval",
        "linear secret sharing",
        "graph secret sharing",
        "monotone span program",
        "CDS",
        "robust secret sharing",
        "verifiable secret sharing",
        "secret sharing",
    ]
    ALL_KEYWORDS += KEYWORDS
    itcr_papers = fetch_arxiv_data(CATEGORIES, 100, logger, DAYS_BACK, KEYWORDS)
    papers += itcr_papers

In [6]:
def collect_csp_papers(papers, ALL_KEYWORDS):
    # Collect some examples on CSP refutation and spectral methods
    KEYWORDS = [
        "matrix concentration",
        "CSP refutation",
        "Kikuchi matrix",
        "spectral method",
        "spectral approach",
        "CSP", 
        "even cover",
        "random walk",
        "mixing time",
    ]
    ALL_KEYWORDS += KEYWORDS
    spectral_papers = fetch_arxiv_data(CATEGORIES, 150, logger, DAYS_BACK, KEYWORDS)
    papers += spectral_papers

In [7]:
def collect_lb_papers(papers, ALL_KEYWORDS):
    # Collect some examples on lower bounds
    KEYWORDS = [
        "rank method",
        "polynomial method",
        "lower bound",
        "entropy method",
        "matrix rigidity",
        "discrepancy",
    ]
    ALL_KEYWORDS += KEYWORDS
    lb_papers = fetch_arxiv_data(CATEGORIES, 75, logger, DAYS_BACK, KEYWORDS)
    papers += lb_papers

In [8]:
def dedup_papers(papers):
    # Deuplicate papers
    already_seen = set()
    deduped = []
    for paper in papers:
        pid = paper["id"].partition("v")[0]
        if pid not in already_seen:
            already_seen.add(pid)
            deduped.append(paper)
    return deduped

In [9]:
def build_dataset(papers, ALL_KEYWORDS):
    collect_coding_papers(papers, ALL_KEYWORDS)
    collect_csp_papers(papers, ALL_KEYWORDS)
    collect_itcr_papers(papers, ALL_KEYWORDS)
    collect_lb_papers(papers, ALL_KEYWORDS)
    collect_locality_papers(papers, ALL_KEYWORDS)
    papers[:] = dedup_papers(papers)

In [10]:
papers = []
build_dataset(papers, ALL_KEYWORDS)
# Computer some statistics about the data set
num_in_category = {"cs.DS": 0, "cs.IT": 0, "cs.CC": 0, "math.CO": 0, "cs.CR": 0}
num_per_keyword = {kw: 0 for kw in ALL_KEYWORDS}

for paper in papers:
    num_in_category[paper['arxiv_category']] += 1
    for kw in ALL_KEYWORDS: 
        if kw in paper['abstract'] or kw in paper['title']:
            num_per_keyword[kw] += 1


print(f"Number of papers retrieved: {len(papers)}")
print(f"Publication date of oldest paper: {papers[-1]['published']}")
pp(num_in_category)
pp(num_per_keyword)

INFO:root:Collected 0 papers so far. Expect 0.06 more batches.
  9%|▊         | 87/1000 [00:35<06:12,  2.45it/s]
INFO:root:Collected 0 papers so far. Expect 0.12 more batches.
 26%|██▌       | 255/1000 [01:32<04:31,  2.74it/s]
INFO:root:Collected 0 papers so far. Expect 0.08 more batches.
 16%|█▋        | 138/844 [00:45<03:52,  3.04it/s]
INFO:root:Collected 0 papers so far. Expect 0.06 more batches.
  9%|▉         | 91/1000 [00:15<02:29,  6.06it/s]
INFO:root:Collected 0 papers so far. Expect 0.12 more batches.
 31%|███       | 176/576 [01:21<03:04,  2.17it/s]


Number of papers retrieved: 501
Publication date of oldest paper: 2023-01-31T00:57:04
{'cs.DS': 95, 'cs.IT': 235, 'cs.CC': 67, 'math.CO': 104, 'cs.CR': 0}
{'erasure coding': 4,
 'list decoding': 25,
 'list recovery': 6,
 'random code': 11,
 'matching vector': 1,
 'matrix concentration': 4,
 'CSP refutation': 1,
 'Kikuchi matrix': 2,
 'spectral method': 12,
 'spectral approach': 4,
 'CSP': 64,
 'even cover': 3,
 'random walk': 47,
 'mixing time': 20,
 'private information retrieval': 92,
 'linear secret sharing': 0,
 'graph secret sharing': 0,
 'monotone span program': 1,
 'CDS': 2,
 'robust secret sharing': 0,
 'verifiable secret sharing': 0,
 'secret sharing': 28,
 'rank method': 0,
 'polynomial method': 2,
 'lower bound': 151,
 'entropy method': 0,
 'matrix rigidity': 0,
 'discrepancy': 3,
 'locally decodable': 16,
 'locally correctable': 9,
 'locally testable': 13,
 'smooth code': 0,
 'locally recoverable': 30}


Issues with the dataset: 
* PIR is overrepresented 
* Kikuchi method and spectral method is not represented well
* The "lower bound" keyword probably adds a lot of noise

Since this is only a preliminary investigation, I will proceed with the dataset as is. The main goal is to determine, roughly, which model is best (SPECTER which is specialized for scientific papers or a generic LLM, and how they compare to BM25), and what level of math removal is best.

In [11]:
for paper in papers:
    if "lower bound" in paper['abstract']:
        print(paper['title'])

List-Decoding Counterexamples Yield Lower Bounds on Mutual Correlated Agreement Error
The Insertion List-Decoding Capacity and an Improved Bound on the Deletion List-Decoding Capacity
Linear Code Conversion in the Merge Regime: General Bounds and Reed-Muller Constructions
Tight Lower Bounds and Optimal Constructions of Locally Repairable Convertible Codes in the Split Regime
List Recovery for Random Low-Rate Linear Codes
The dimensions of Schur squares of HRS codes
On the Capacity of Distinguishable Synthetic Identity Generation under Face Verification
The Random Subsequence Model and Uniform Codes for the Deletion Channel
Error Exponents for Randomised List Decoding
Fourier Sparsity of Delta Functions and Matching Vector PIRs
Tight Lower Bounds on the Bandwidth Cost of MDS Convertible Codes in the Split Regime
Combinatorial Bounds for List Recovery via Discrete Brascamp--Lieb Inequalities
List Decoding Reed--Solomon Codes in the Lee, Euclidean, and Other Metrics
Strong Refutation of R

## Evaluations
As stated, we will evaluate the 3 levels of math removal for 3 different models, BM25, SPECTER, and all-MiniLM-L12-v2. We begin with the coarse removal, since that is the data already collected above, and then we will collect data on the other two levels of math removal.

Code in this section is generated by GPT5.6 Terra

In [12]:
from copy import deepcopy

import requests

from src.extract_intro import IntroExtractionError, get_intro_text

COARSENESS_LEVELS = ("no_math", "coarse", "fine")


def build_matched_corpora(papers):
    """Re-extract every selected paper at every math-normalization level."""
    variants = {level: {} for level in COARSENESS_LEVELS}

    with requests.Session() as session:
        for level in COARSENESS_LEVELS:
            for paper in tqdm(papers, desc=f"Extracting {level}"):
                try:
                    introduction = get_intro_text(session, paper["id"], level)
                except IntroExtractionError:
                    continue

                if introduction:
                    variant = deepcopy(paper)
                    variant["introduction"] = introduction
                    variants[level][variant["id"]] = variant

    # Every model and normalization must see the same papers.
    shared_ids = set.intersection(*(set(variant) for variant in variants.values()))
    ordered_ids = [paper["id"] for paper in papers if paper["id"] in shared_ids]
    matched = {
        level: [variants[level][paper_id] for paper_id in ordered_ids]
        for level in COARSENESS_LEVELS
    }

    print(f"Matched evaluation corpus: {len(ordered_ids)} papers")
    print({level: len(variants[level]) for level in COARSENESS_LEVELS})
    return matched


papers_by_coarseness = build_matched_corpora(papers)


Extracting fine: 100%|██████████| 502/502 [00:27<00:00, 17.97it/s]

Matched evaluation corpus: 502 papers
{'no_math': 502, 'coarse': 502, 'fine': 502}


In [13]:
import numpy as np
import torch
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, AutoTokenizer


TEXT_SCOPES = {
    "title": ("title",),
    "title_abstract": ("title", "abstract"),
    "title_abstract_intro": ("title", "abstract", "introduction"),
}

# Change this, then re-run this cell and the nearest-neighbours cell.
EMBEDDING_TEXT_SCOPE = "title_abstract_intro"

def paper_text(paper, text_scope=EMBEDDING_TEXT_SCOPE):
    """Build the text representation used for one embedding experiment."""
    if text_scope not in TEXT_SCOPES:
        raise ValueError(f"Unknown text scope {text_scope!r}; choose from {tuple(TEXT_SCOPES)}")
    return "\n\n".join(paper[field] for field in TEXT_SCOPES[text_scope])


class BM25Index:
    def __init__(self, documents, k1=1.5, b=0.75):
        self.vectorizer = CountVectorizer(token_pattern=r"(?u)\b\w+\b", lowercase=True)
        self.term_counts = self.vectorizer.fit_transform(documents).tocsr()
        self.k1 = k1
        self.b = b
        self.document_lengths = np.asarray(self.term_counts.sum(axis=1)).ravel()
        self.average_length = self.document_lengths.mean()
        document_frequency = np.asarray((self.term_counts > 0).sum(axis=0)).ravel()
        num_documents = self.term_counts.shape[0]
        self.idf = np.log(1 + (num_documents - document_frequency + 0.5) / (document_frequency + 0.5))

    def score(self, query):
        query_terms = self.vectorizer.transform([query]).indices
        if not len(query_terms):
            return np.zeros(self.term_counts.shape[0])

        frequencies = self.term_counts[:, query_terms].toarray()
        length_penalty = self.k1 * (1 - self.b + self.b * self.document_lengths / self.average_length)
        return (
            self.idf[query_terms]
            * (frequencies * (self.k1 + 1) / (frequencies + length_penalty[:, None]))
        ).sum(axis=1)


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)
_specter_tokenizer = None
_specter_model = None
_minilm_model = None
_nomic_model = None


def embed_specter(documents, batch_size=8):
    global _specter_tokenizer, _specter_model
    if _specter_model is None:
        _specter_tokenizer = AutoTokenizer.from_pretrained("allenai/specter")
        _specter_model = AutoModel.from_pretrained("allenai/specter").to(DEVICE)
        _specter_model.eval()

    vectors = []

    for start in tqdm(range(0, len(documents), batch_size), desc="Embedding with SPECTER"):
        batch = documents[start : start + batch_size]
        inputs = _specter_tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt")
        inputs = {name: value.to(DEVICE) for name, value in inputs.items()}
        with torch.no_grad():
            vectors.append(_specter_model(**inputs).last_hidden_state[:, 0].cpu().numpy())

    return normalize(np.vstack(vectors))


def embed_minilm(documents, batch_size=32):
    global _minilm_model
    if _minilm_model is None:
        _minilm_model = SentenceTransformer("sentence-transformers/all-MiniLM-L12-v2", device=DEVICE)
    return _minilm_model.encode(
        documents,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    )


def embed_nomic(documents, batch_size=2):
    global _nomic_model
    if _nomic_model is None:
        _nomic_model = SentenceTransformer(
            "nomic-ai/nomic-embed-text-v1.5",
            trust_remote_code=True,
            device=DEVICE,
        )
        _nomic_model.max_seq_length = 2048

    clustering_documents = [f"clustering: {document}" for document in documents]
    return _nomic_model.encode(
        clustering_documents,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    )


def build_evaluation_indexes(papers_by_coarseness, model, text_scope=EMBEDDING_TEXT_SCOPE):
    indexes = {}
    for level, level_papers in papers_by_coarseness.items():
        documents = [paper_text(paper, text_scope) for paper in level_papers]
        print(f"Building indexes for {level} / {text_scope} ({len(documents)} papers)")
        if (model == "bm25"):
            embeddings = BM25Index(documents)
        elif model == "specter":
            embeddings = embed_specter(documents)
        elif model == "minilm":
            embeddings = embed_minilm(documents)
        else:
            embeddings = embed_nomic(documents, batch_size=4)
        indexes[level] = {
            model: embeddings
        }
        # Should not embed the same thing 3 times
        if "intro" not in text_scope:
            return indexes
    return indexes

evaluation_indexes = build_evaluation_indexes(papers_by_coarseness, "bm25", EMBEDDING_TEXT_SCOPE)

cuda
Building indexes for no_math / title_abstract_intro (502 papers)
Building indexes for coarse / title_abstract_intro (502 papers)
Building indexes for fine / title_abstract_intro (502 papers)


In [14]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

cuda


In [20]:
from IPython.display import display


def _paper_position(paper_id_or_title, level):
    level_papers = papers_by_coarseness[level]
    exact_matches = [
        index
        for index, paper in enumerate(level_papers)
        if paper_id_or_title in (paper["id"], paper["title"])
    ]
    if len(exact_matches) == 1:
        return exact_matches[0]

    title_matches = [
        index
        for index, paper in enumerate(level_papers)
        if paper_id_or_title.lower() in paper["title"].lower()
    ]
    if len(title_matches) == 1:
        return title_matches[0]
    if not title_matches:
        raise KeyError(f"No paper matched {paper_id_or_title!r}")

    matches = [level_papers[index]["title"] for index in title_matches[:10]]
    raise ValueError(f"Multiple papers matched {paper_id_or_title!r}: {matches}")


def nearest_neighbours(
    paper_id_or_title, model="specter", coarseness="coarse",
    text_scope=EMBEDDING_TEXT_SCOPE, k=10
):
    """Display the nearest papers for an arXiv ID, exact title, or unique title fragment."""
    if coarseness not in papers_by_coarseness:
        raise ValueError(f"Unknown coarseness {coarseness!r}; choose from {COARSENESS_LEVELS}")
    if model not in {"bm25", "specter", "minilm", "nomic"}:
        raise ValueError("model must be 'bm25', 'specter', 'minilm', or 'nomic'")
    if text_scope != EMBEDDING_TEXT_SCOPE:
        raise ValueError(
            f"Indexes were built for {EMBEDDING_TEXT_SCOPE!r}; re-run the indexing cell for {text_scope!r}."
        )

    position = _paper_position(paper_id_or_title, coarseness)
    level_papers = papers_by_coarseness[coarseness]
    documents = [paper_text(paper, text_scope) for paper in level_papers]
    index = evaluation_indexes[coarseness][model]

    if model == "bm25":
        scores = index.score(documents[position])
    else:
        scores = index @ index[position]

    ranked_positions = [
        candidate
        for candidate in np.argsort(-scores)
        if candidate != position
    ][:k]
    query = level_papers[position]
    print(f"Query: {query['title']} ({query['id']}) | text: {text_scope}")
    results = pd.DataFrame(
        [
            {
                "rank": rank,
                "score": round(float(scores[candidate]), 4),
                "title": level_papers[candidate]["title"],
                "arxiv_id": level_papers[candidate]["id"],
                "category": level_papers[candidate]["arxiv_category"],
                "url": level_papers[candidate]["url"],
            }
            for rank, candidate in enumerate(ranked_positions, start=1)
        ]
    )
    display(results)
    return results

# Example: replace this with an arXiv ID or a distinctive title fragment.
model = "nomic"
# evaluation_indexes = build_evaluation_indexes(papers_by_coarseness, model, EMBEDDING_TEXT_SCOPE)
_ = nearest_neighbours("2404.06513v2", model=model, coarseness='no_math')
_ = nearest_neighbours("2404.06513v2", model=model, coarseness='coarse')
_ = nearest_neighbours("2404.06513v2", model=model, coarseness='fine')



Query: Exponential Lower Bounds for Smooth 3-LCCs and Sharp Bounds for Designs (2404.06513v2) | text: title_abstract_intro


,rank,score,title,arxiv_id,category,url
0,1,0.9278,An Exponential Lower Bound for Linear 3-Query ...,2311.00558v1,cs.CC,https://arxiv.org/abs/2311.00558v1
1,2,0.9065,Near-Tight Bounds for 3-Query Locally Correcta...,2404.05864v1,cs.IT,https://arxiv.org/abs/2404.05864v1
2,3,0.8688,Exponential Lower Bounds for 2-query Relaxed L...,2602.20278v2,cs.IT,https://arxiv.org/abs/2602.20278v2
3,4,0.8618,Improved Lower Bounds for all Odd-Query Locall...,2411.14361v1,cs.CC,https://arxiv.org/abs/2411.14361v1
4,5,0.8597,A Near-Cubic Lower Bound for 3-Query Locally D...,2308.15403v1,cs.CC,https://arxiv.org/abs/2308.15403v1
5,6,0.8484,Relaxed vs. Full Local Decodability with Few Q...,2511.02633v2,cs.CC,https://arxiv.org/abs/2511.02633v2
6,7,0.8456,Good Locally Testable Codes with Small Alphabe...,2512.16082v2,cs.CC,https://arxiv.org/abs/2512.16082v2
7,8,0.8432,A $k^{\frac{q}{q-2}}$ Lower Bound for Odd Quer...,2411.14276v2,cs.CC,https://arxiv.org/abs/2411.14276v2
8,9,0.8378,When Relaxation Does Not Help: RLDCs with Smal...,2603.03717v1,cs.IT,https://arxiv.org/abs/2603.03717v1
9,10,0.8368,Low Degree Local Correction Over the Boolean Cube,2411.07374v2,cs.CC,https://arxiv.org/abs/2411.07374v2


Query: Exponential Lower Bounds for Smooth 3-LCCs and Sharp Bounds for Designs (2404.06513v2) | text: title_abstract_intro


,rank,score,title,arxiv_id,category,url
0,1,0.9292,An Exponential Lower Bound for Linear 3-Query ...,2311.00558v1,cs.CC,https://arxiv.org/abs/2311.00558v1
1,2,0.9056,Near-Tight Bounds for 3-Query Locally Correcta...,2404.05864v1,cs.IT,https://arxiv.org/abs/2404.05864v1
2,3,0.8684,Exponential Lower Bounds for 2-query Relaxed L...,2602.20278v2,cs.IT,https://arxiv.org/abs/2602.20278v2
3,4,0.8614,A Near-Cubic Lower Bound for 3-Query Locally D...,2308.15403v1,cs.CC,https://arxiv.org/abs/2308.15403v1
4,5,0.8598,Improved Lower Bounds for all Odd-Query Locall...,2411.14361v1,cs.CC,https://arxiv.org/abs/2411.14361v1
5,6,0.8487,Relaxed vs. Full Local Decodability with Few Q...,2511.02633v2,cs.CC,https://arxiv.org/abs/2511.02633v2
6,7,0.8466,Good Locally Testable Codes with Small Alphabe...,2512.16082v2,cs.CC,https://arxiv.org/abs/2512.16082v2
7,8,0.8454,A $k^{\frac{q}{q-2}}$ Lower Bound for Odd Quer...,2411.14276v2,cs.CC,https://arxiv.org/abs/2411.14276v2
8,9,0.8387,Low Degree Local Correction Over the Boolean Cube,2411.07374v2,cs.CC,https://arxiv.org/abs/2411.07374v2
9,10,0.8346,When Relaxation Does Not Help: RLDCs with Smal...,2603.03717v1,cs.IT,https://arxiv.org/abs/2603.03717v1


Query: Exponential Lower Bounds for Smooth 3-LCCs and Sharp Bounds for Designs (2404.06513v2) | text: title_abstract_intro


,rank,score,title,arxiv_id,category,url
0,1,0.9305,An Exponential Lower Bound for Linear 3-Query ...,2311.00558v1,cs.CC,https://arxiv.org/abs/2311.00558v1
1,2,0.9054,Near-Tight Bounds for 3-Query Locally Correcta...,2404.05864v1,cs.IT,https://arxiv.org/abs/2404.05864v1
2,3,0.8580,A Near-Cubic Lower Bound for 3-Query Locally D...,2308.15403v1,cs.CC,https://arxiv.org/abs/2308.15403v1
3,4,0.8577,Exponential Lower Bounds for 2-query Relaxed L...,2602.20278v2,cs.IT,https://arxiv.org/abs/2602.20278v2
4,5,0.8511,Improved Lower Bounds for all Odd-Query Locall...,2411.14361v1,cs.CC,https://arxiv.org/abs/2411.14361v1
5,6,0.8378,A $k^{\frac{q}{q-2}}$ Lower Bound for Odd Quer...,2411.14276v2,cs.CC,https://arxiv.org/abs/2411.14276v2
6,7,0.8360,Relaxed vs. Full Local Decodability with Few Q...,2511.02633v2,cs.CC,https://arxiv.org/abs/2511.02633v2
7,8,0.8248,When Relaxation Does Not Help: RLDCs with Smal...,2603.03717v1,cs.IT,https://arxiv.org/abs/2603.03717v1
8,9,0.8224,Good Locally Testable Codes with Small Alphabe...,2512.16082v2,cs.CC,https://arxiv.org/abs/2512.16082v2
9,10,0.8129,Low Degree Local Correction Over the Boolean Cube,2411.07374v2,cs.CC,https://arxiv.org/abs/2411.07374v2


## Summary of Evaluations. 
The two cells below were run for all combinations of math removal (no_math, coarse, fine), embedding model (bm25, specter, minilm, nomic), and text scope (title, title + abstract, title + abstract + intro). I manually evaluated the accuracy of the top 10 nearest neighbours of a few specially chosen papers. Not all runs of the experiment will be visible as output in the cells below, but a summary is provided here.

### Title only
**Case study 1: 2404.06513v2 (Exponential Lower Bounds for Smooth 3-LCCs and Sharp Bounds for Designs)**

**SPECTER**
* Top paper is [2309.03676v1](https://arxiv.org/abs/2309.03676v1) which is not very related.
* Remaining papers are pretty much unrelated, such as [2507.08693v1](https://arxiv.org/abs/2507.08693v1)

**MiniLM**
* Top paper is actually correct! [2311.00558v1](https://arxiv.org/abs/2311.00558v1) is the immediate predeccessor of this work.
* Lower ranked papers are still mostly not accurate, i.e. [2607.07576v1](https://arxiv.org/abs/2607.07576v1), but it does catch [2404.05864v1](https://arxiv.org/abs/2404.05864v1) and [2308.15403v1](https://arxiv.org/abs/2308.15403v1). Seems to put too much weight on the word lower bound.

**Nomic**
* Top paper is correct, and ranked with higher confidence than miniLM.
* Lower ranked papers are a little unrelated, better than SPECTER but worse than miniLM. [2607.06551v1](https://arxiv.org/abs/2607.06551v1)

**BM25** 
* Incredibly fast to compute indices
* Top paper is correct again. 
* Lower ranked papers are not even close, [2607.04373v1](https://arxiv.org/abs/2607.04373v1), [2607.12449v1](https://arxiv.org/abs/2607.12449v1). This was pretty much expected.

**Case study 2: 2509.06209v1 (Efficient Catalytic Graph Algorithms)** 

**SPECTER**
* Ranked [2602.14320v2](https://arxiv.org/abs/2602.14320v2) as third when it probably should be first. 
* Gets a lot of graph algorithms, which is good, but doesn't do much with catalyic complexity. 
* Lowest ranked paper [2604.05976v2](https://arxiv.org/abs/2604.05976v2) is unrelated.

**MiniLM**
* Ranks [2602.14320v2](https://arxiv.org/abs/2602.14320v2) as most relevant 
* Also gets a lot of papers on graph algorithms, which I feel are about the same relevance as SPECTER's papers.
* Lowest ranked paper [2512.24434v4](https://arxiv.org/abs/2512.24434v4) is a survey on non-backtracking random walks, which is silghtly more related than the lowest ranked paper by SPECTER but not by much

**Nomic**
* Top few papers are the same as MiniLM but with more confidence 
* Lower ranked papers are strange, like [2607.08261v1](https://arxiv.org/abs/2607.08261v1) which is quite unrelated, except that it's a graph algorithm

**BM25**
* Found a second paper about catalytic computing! This was expected, since BM25 will give the relatively rare catalytic keyword a lot of weight
* Lower ranked papers are least relevant of all models, i.e. [2602.14243v2](https://arxiv.org/abs/2602.14243v2)

### Title + Abstract
**Case study 1: 2404.06513v2 (Exponential Lower Bounds for Smooth 3-LCCs and Sharp Bounds for Designs)**

**SPECTER**
* Top 5 papers are all very good and in the right order!
* I would put [2308.15403v1](https://arxiv.org/abs/2308.15403v1) currently ranked 7, above [2603.03717v1](https://arxiv.org/abs/2603.03717v1) ranked 6, and maybe even above [2404.05864v1](https://arxiv.org/abs/2404.05864v1). 
* The lowest 2 papers [2512.07343v2](https://arxiv.org/abs/2512.07343v2) and [2508.13553v1](https://arxiv.org/abs/2508.13553v1) are about LRCs but they aren't about lower bounds, so they're not very relevant
* Kind of slow, took about 3 minutes to finish embedding.

**MiniLM**
* Mostly very good results. I would say the top 2 papers are very good, and slightly more accurate than SPECTER's. 
* Main issue is that [2411.14276v2](https://arxiv.org/abs/2411.14276v2) is ranked too low, below some stuff about RLDCs
* Embeddings finished quite quickly, in about 15s

**Nomic**
* Top 2 papers are both exponential lower bounds for LCCs, which is acceptable.
* Has some issues in the lower rankings. [2512.16082v2](https://arxiv.org/abs/2512.16082v2) isn't about LDCs, LCCs, or lower bounds, so it definitely shouldn't be so high up.
* [2411.14276v2](https://arxiv.org/abs/2411.14276v2) is ranked 10th, below [2403.20305v2](https://arxiv.org/abs/2403.20305v2) which is much less relevant
* Total embedding time took about 3 minutes

**BM25**
* Top papers are actually very relevant, more or less the same as in SPECTER.
* Confidence of the top ranked paper is very high, while 6th ranked paper has low confidence 
* Includes an irrelevant paper ranked 10th [2607.07257v1](https://arxiv.org/abs/2607.07257v1) when it would have been better to include something about RLDCs

**Case study 2: 2509.06209v1 (Efficient Catalytic Graph Algorithms)** 

**SPECTER**
* Overall very poor results. Top ranked paper [2508.14238v1](https://arxiv.org/abs/2508.14238v1) isn't about graph algorithms, and only very tangentially about simulating random walks. While the paper [2607.09475v1](https://arxiv.org/abs/2607.09475v1) which is very relevant is quite low down. 
* Misses the paper [2602.14320v2](https://arxiv.org/abs/2602.14320v2), which should be the most relevant. 

**MiniLM**
* Correctly ranks [2602.14320v2](https://arxiv.org/abs/2602.14320v2) at first place
* Ranks [2607.09475v1](https://arxiv.org/abs/2607.09475v1) at 8th place, while it should be higher up
* Gets one paper about sublinear time graph algorithms (original paper is more about sublinear space, but this is close enough). Other papers are irrelevant

**Nomic**
* Best results for this paper. Ranks both of the other catalytic papers right at the top.
* Includes [2606.13583v1](https://arxiv.org/abs/2606.13583v1) which is a sublinear space streaming algorithm, but ranks it 10th when it should be higher
* Other papers are slightly more relevant than MiniLM

**BM25**
* Ranks some paper about estimating random walks [2504.16481v4](https://arxiv.org/abs/2504.16481v4) above the catalytic papers
* Finds some nice papers sublinear time [2601.05883v1](https://arxiv.org/abs/2601.05883v1) and sublinear space [2511.04343v1](https://arxiv.org/abs/2511.04343v1) papers, but includes other irrelevant ones

### Title + Abstract + Intro
In this section I only evaluate the Nomic and BM25 Embeddings, because SPECTER and MiniLM have small context windows which barely fits the title and abstract.

**Case study 1: 2404.06513v2 (Exponential Lower Bounds for Smooth 3-LCCs and Sharp Bounds for Designs)**

**BM25**

no_math: mixes in some papers which are not so relevant, like [2603.03717v1](https://arxiv.org/abs/2603.03717v1) (ranked 3rd) and ranks [2401.11590v2](https://arxiv.org/abs/2401.11590v2) too low (10th) but overall good performance

coarse: same as no_math. Replaced keywords probably appear very often and hence are not given much weight

fine: same as no_math. Compactified math expressions are probably unique in the entire corpus, since rarely do people write the same thing in exactly the same way.

**Nomic**

no_math: Good performance on top 5 papers. Mixes in [2512.16082v2](https://arxiv.org/abs/2512.16082v2) in rank 7, which shouldn't be in the top 10

coarse: Not much difference from no_math

fine: Improved rankings. Same papers are ranked in the top 10 but reordered in a more sensible way. 

**Case study 2: 2509.06209v1 (Efficient Catalytic Graph Algorithms)** 

**BM25**

no_math: Correctly ranks the two catalytic computing papers, but then includes a bunch of papers on codes and CSPs. I'm not sure why this is the case, but those are not at all related. 

**Nomic**

no_math: Correctly identifies the top two papers, and finds some good sublinear time / space papers as well. I would say it does slightly better than Nomic with just title + abstract

coarse: Not much difference from no_math

fine: Not much difference from no_math.


In [16]:
papers[180]

{'title': 'Efficient Catalytic Graph Algorithms',
 'authors': ['James Cook', 'Edward Pyne'],
 'published': '2025-09-07T21:14:13',
 'abstract': "We give fast, simple, and implementable catalytic logspace algorithms for two fundamental graph problems.\n  First, a randomized catalytic algorithm for $s\\to t$ connectivity running in $\\widetilde{O}(nm)$ time, and a deterministic catalytic algorithm for the same running in $\\widetilde{O}(n^3 m)$ time. The former algorithm is the first algorithmic use of randomization in $\\mathsf{CL}$. The algorithm uses one register per vertex and repeatedly ``pushes'' values along the edges in the graph.\n  Second, a deterministic catalytic algorithm for simulating random walks which in $\\widetilde{O}( m T^2 / \\varepsilon )$ time estimates the probability a $T$-step random walk ends at a given vertex within $\\varepsilon$ additive error. The algorithm uses one register for each vertex and increments it at each visit to ensure repeated visits follow dif